In [12]:
from __future__ import annotations

from decimal import Decimal, ROUND_HALF_UP
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine


# =========================
# 1) 설정
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

SCHEMA = "d1_machine_log"
TABLES_VISION = ["Vision1_machine_log", "Vision2_machine_log"]

END_DAY = "20260130"

SIGNAL_GOOD = "검사 양품 신호 출력"
SIGNAL_BAD  = "검사 불량 신호 출력"

# Vision2에 느낌표가 붙을 수 있으니 prefix로 매칭
SCAN_PREFIX = "바코드 스캔 신호 수신"

# ✅ 500초 이상 제거
MAX_SEC = 500.0  # delta >= 500 제외


# =========================
# 2) DB 연결
# =========================
def make_engine(cfg: dict) -> Engine:
    url = (
        f"postgresql+psycopg2://{cfg['user']}:{cfg['password']}"
        f"@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )


# =========================
# 3) 유틸
# =========================
def parse_time_to_seconds(t: str) -> Optional[float]:
    """'HH:MI:SS.xx' 또는 'HH:MI:SS' -> 초(float). 파싱 실패 시 None."""
    if t is None:
        return None
    s = str(t).strip()
    if not s:
        return None

    parts = s.split(":")
    if len(parts) != 3:
        return None

    try:
        hh = int(parts[0])
        mm = int(parts[1])
        sec_part = parts[2]

        if "." in sec_part:
            ss_str, frac_str = sec_part.split(".", 1)
            ss = int(ss_str)
            frac = float("0." + "".join(ch for ch in frac_str if ch.isdigit()))
        else:
            ss = int(sec_part)
            frac = 0.0

        return hh * 3600 + mm * 60 + ss + frac
    except Exception:
        return None


def round_half_up(x: float, ndigits: int = 2) -> float:
    """ROUND_HALF_UP 방식 반올림"""
    q = Decimal("1").scaleb(-ndigits)  # 10^-ndigits
    return float(Decimal(str(x)).quantize(q, rounding=ROUND_HALF_UP))


def tukey_5vals_with_range(values: List[float]) -> Dict[str, Optional[object]]:
    """
    5값(whisker 기반) + upper outlier range:
      - lower_outlier = lower whisker (Tukey, 데이터 기반)
      - upper_outlier = upper whisker (Tukey, 데이터 기반)
      - upper_outlier_max = whisker 초과(outlier) 최대값 (없으면 None)
      - upper_outlier_range = "upper_outlier~upper_outlier_max" 문자열
    """
    if not values:
        return {
            "lower_outlier": None, "q1": None, "median": None, "q3": None, "upper_outlier": None,
            "upper_outlier_max": None, "upper_outlier_range": None
        }

    arr = np.asarray(values, dtype=float)
    arr.sort()

    q1  = float(np.quantile(arr, 0.25, method="linear"))
    med = float(np.quantile(arr, 0.50, method="linear"))
    q3  = float(np.quantile(arr, 0.75, method="linear"))
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    # fence 안쪽(inlier)의 "실제 데이터"로 whisker 결정
    inlier = arr[(arr >= lower_fence) & (arr <= upper_fence)]
    if inlier.size == 0:
        lw = float(arr.min())
        uw = float(arr.max())
    else:
        lw = float(inlier.min())
        uw = float(inlier.max())

    # upper whisker 초과(outlier) 최대값
    upper_outliers = arr[arr > uw]
    u_max = float(upper_outliers.max()) if upper_outliers.size > 0 else None

    # 반올림(ROUND_HALF_UP, 2자리)
    lw_r  = round_half_up(lw, 2)
    q1_r  = round_half_up(q1, 2)
    med_r = round_half_up(med, 2)
    q3_r  = round_half_up(q3, 2)
    uw_r  = round_half_up(uw, 2)

    if u_max is not None:
        u_max_r = round_half_up(u_max, 2)
        u_range = f"{uw_r:.2f}~{u_max_r:.2f}"
    else:
        u_max_r = None
        u_range = f"{uw_r:.2f}~{uw_r:.2f}"

    return {
        "lower_outlier": lw_r,
        "q1": q1_r,
        "median": med_r,
        "q3": q3_r,
        "upper_outlier": uw_r,
        "upper_outlier_max": u_max_r,
        "upper_outlier_range": u_range,
    }


# =========================
# 4) 데이터 로드
# =========================
def fetch_events(engine: Engine, schema: str, table: str, end_day: str) -> pd.DataFrame:
    # ✅ 대/소문자 혼합 식별자 보호
    fqn = f'"{schema}"."{table}"'

    # ✅ SCAN은 prefix로 필터 (수신 / 수신! 모두 포함)
    sql = text(f"""
        SELECT
            end_time,
            contents
        FROM {fqn}
        WHERE end_day = :end_day
          AND end_time IS NOT NULL
          AND (
                contents IN (:good, :bad)
                OR contents LIKE :scan_like
              )
        ORDER BY end_time ASC
    """)

    with engine.begin() as conn:
        df = pd.read_sql(
            sql, conn,
            params={
                "end_day": end_day,
                "good": SIGNAL_GOOD,
                "bad": SIGNAL_BAD,
                "scan_like": f"{SCAN_PREFIX}%"
            }
        )
    return df


# =========================
# 5) 페어링 & 지연시간 계산
# =========================
def compute_scan_delay_seconds(df_events: pd.DataFrame) -> List[float]:
    """
    (GOOD/BAD 신호) -> 다음 SCAN 페어링으로 (SCAN - SIGNAL) seconds 리스트 반환
    필터:
      - delta <= 0 제외
      - delta >= 500 제외
    """
    df = df_events.copy()
    df["t_sec"] = df["end_time"].map(parse_time_to_seconds)
    df = df.dropna(subset=["t_sec"])
    df = df.sort_values("t_sec", ascending=True).reset_index(drop=True)

    pending_signal: Optional[float] = None
    out: List[float] = []

    for _, r in df.iterrows():
        c = str(r["contents"])
        t = float(r["t_sec"])

        if c == SIGNAL_GOOD or c == SIGNAL_BAD:
            pending_signal = t

        elif c.startswith(SCAN_PREFIX):
            if pending_signal is None:
                continue

            delta = t - pending_signal  # SCAN - SIGNAL
            pending_signal = None

            if delta <= 0:
                continue
            if delta >= MAX_SEC:  # ✅ 500초 이상 제거
                continue

            out.append(round_half_up(delta, 2))

    return out


# =========================
# 6) 실행 (DF 출력)
# =========================
engine = make_engine(DB_CONFIG)

all_rows = []
per_table_values: Dict[str, List[float]] = {}

for t in TABLES_VISION:
    df_e = fetch_events(engine, SCHEMA, t, END_DAY)
    vals = compute_scan_delay_seconds(df_e)
    per_table_values[t] = vals
    for v in vals:
        all_rows.append({"table": t, "delay_sec": v})

df_all = pd.DataFrame(all_rows)

# (1) 페어 결과 DF 출력(원하면 head만 보자)
print("=== Pairing Result (df_all) ===")
display(df_all)  # 주피터에서 전체 테이블 형태로 출력

# (2) 5값 요약 DF (테이블별 + 통합)
summary_rows = []
for t, vals in per_table_values.items():
    s = tukey_5vals_with_range(vals)
    summary_rows.append({"table": t, "n": len(vals), **s})

vals_total = df_all["delay_sec"].tolist() if not df_all.empty else []
s_total = tukey_5vals_with_range(vals_total)
summary_rows.append({"table": "ALL_VISION", "n": len(vals_total), **s_total})

df_summary = pd.DataFrame(summary_rows)

# 컬럼 정렬
cols = [
    "table", "n",
    "lower_outlier", "q1", "median", "q3", "upper_outlier",
    "upper_outlier_max", "upper_outlier_range"
]
df_summary = df_summary[[c for c in cols if c in df_summary.columns]]

print("=== Summary (df_summary) ===")
display(df_summary)

=== Pairing Result (df_all) ===


,table,delay_sec
0,Vision1_machine_log,10.77
1,Vision1_machine_log,17.62
2,Vision1_machine_log,59.66
3,Vision1_machine_log,10.83
4,Vision1_machine_log,18.82
...,...,...
7589,Vision2_machine_log,12.84
7590,Vision2_machine_log,15.45
7591,Vision2_machine_log,11.39
7592,Vision2_machine_log,17.35


=== Summary (df_summary) ===


,table,n,lower_outlier,q1,median,q3,upper_outlier,upper_outlier_max,upper_outlier_range
0,Vision1_machine_log,3649,6.70,11.09,15.67,18.26,28.83,349.21,28.83~349.21
1,Vision2_machine_log,3945,5.67,9.68,14.08,17.64,29.58,384.94,29.58~384.94
2,ALL_VISION,7594,5.67,10.73,14.78,18.01,28.83,384.94,28.83~384.94


In [14]:
# =========================
# (추가) Vision: upper_outlier_range 구간 데이터만으로 boxplot 5값 재산출
# - range = [upper_outlier, upper_outlier_max] (양끝 포함)
# - subset으로 Tukey 5값(whisker 기반) 재계산
# =========================

def tukey_5vals_whisker(values: List[float]) -> Dict[str, Optional[float]]:
    """
    lower_outlier, q1, median, q3, upper_outlier
    = Tukey whisker(데이터 기반) 5값
    """
    if not values:
        return {"lower_outlier": None, "q1": None, "median": None, "q3": None, "upper_outlier": None}

    arr = np.asarray(values, dtype=float)
    arr.sort()

    q1  = float(np.quantile(arr, 0.25, method="linear"))
    med = float(np.quantile(arr, 0.50, method="linear"))
    q3  = float(np.quantile(arr, 0.75, method="linear"))
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    inlier = arr[(arr >= lower_fence) & (arr <= upper_fence)]
    if inlier.size == 0:
        lw = float(arr.min())
        uw = float(arr.max())
    else:
        lw = float(inlier.min())
        uw = float(inlier.max())

    return {
        "lower_outlier": round_half_up(lw, 2),
        "q1": round_half_up(q1, 2),
        "median": round_half_up(med, 2),
        "q3": round_half_up(q3, 2),
        "upper_outlier": round_half_up(uw, 2),
    }


rows = []

for t in TABLES_VISION:
    vals = per_table_values.get(t, [])
    if not vals:
        rows.append({"table": t, "range": None, "n_range": 0,
                     "lower_outlier": None, "q1": None, "median": None, "q3": None, "upper_outlier": None,
                     "upper_outlier_max": None, "upper_outlier_range": None})
        continue

    r = df_summary.loc[df_summary["table"] == t]
    if r.empty or pd.isna(r["upper_outlier"].iloc[0]) or pd.isna(r["upper_outlier_max"].iloc[0]):
        rows.append({"table": t, "range": None, "n_range": 0,
                     "lower_outlier": None, "q1": None, "median": None, "q3": None, "upper_outlier": None,
                     "upper_outlier_max": None, "upper_outlier_range": None})
        continue

    lo = float(r["upper_outlier"].iloc[0])       # 원본 upper whisker
    hi = float(r["upper_outlier_max"].iloc[0])   # 원본 outlier max

    # ✅ upper_outlier_range 구간 데이터만 추출 (lo~hi)
    range_vals = [v for v in vals if (v >= lo and v <= hi)]

    s = tukey_5vals_whisker(range_vals)

    rows.append({
        "table": t,
        "range": f"{lo:.2f}~{hi:.2f}",
        "n_range": len(range_vals),
        **s,
        "upper_outlier_max": round_half_up(hi, 2),
        "upper_outlier_range": f"{s['upper_outlier']:.2f}~{round_half_up(hi,2):.2f}" if s["upper_outlier"] is not None else None,
    })

# 통합(ALL_VISION)도 같이 계산
vals_total = []
for t in TABLES_VISION:
    vals_total.extend(per_table_values.get(t, []))

r_all = df_summary.loc[df_summary["table"] == "ALL_VISION"]
if not r_all.empty and not pd.isna(r_all["upper_outlier"].iloc[0]) and not pd.isna(r_all["upper_outlier_max"].iloc[0]):
    lo = float(r_all["upper_outlier"].iloc[0])
    hi = float(r_all["upper_outlier_max"].iloc[0])
    range_vals = [v for v in vals_total if (v >= lo and v <= hi)]
    s = tukey_5vals_whisker(range_vals)
    rows.append({
        "table": "ALL_VISION",
        "range": f"{lo:.2f}~{hi:.2f}",
        "n_range": len(range_vals),
        **s,
        "upper_outlier_max": round_half_up(hi, 2),
        "upper_outlier_range": f"{s['upper_outlier']:.2f}~{round_half_up(hi,2):.2f}" if s["upper_outlier"] is not None else None,
    })

df_upper_outlier_range_box = pd.DataFrame(rows)
df_upper_outlier_range_box

,table,range,n_range,lower_outlier,q1,median,q3,upper_outlier,upper_outlier_max,upper_outlier_range
0,Vision1_machine_log,28.83~349.21,295,28.83,33.42,36.12,40.56,49.58,349.21,49.58~349.21
1,Vision2_machine_log,29.58~384.94,155,29.58,32.15,34.33,46.51,67.77,384.94,67.77~384.94
2,ALL_VISION,28.83~384.94,450,28.83,32.93,35.64,42.76,57.37,384.94,57.37~384.94
